# 1. How much faster is CUDA than OpenCL?

This notebook runs the same seismic calculations on the two backends of PyFWI-CUDA, on the same GPU, and
times them. You will see:

1. that the backend is chosen by one entry of `inpa`: the rest of the code does not change;
2. the time of a forward modelling and of a gradient on four engines (two backends × two numeric modes) and
   three model sizes;
3. how many times faster CUDA is when both backends do exactly the same mathematics, and when each runs its
   default.

It takes about 4 minutes on a GeForce RTX 3060, most of them on OpenCL. Your numbers will not be exactly
those of the [final document](../../docs/cuda-migration/cuda-vs-opencl.md), measured more carefully (more
repetitions, larger models), but the picture should be the same.

**A few words.** A *forward modelling* simulates the waves of every seismic source through a model of the
subsurface, one small time step after another. A *gradient* runs a second simulation backwards in time to find
how the model should change; an inversion repeats the two again and again. The *numeric mode* is `legacy`
(the arithmetic of the original PyFWI 0.1.10, bugs included) or `fixed` (the bugs corrected). OpenCL uses
`legacy` by default and CUDA `fixed`, but either backend runs either mode.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter

from comparison import (  # the code shared by the notebooks of this folder: comparison.py
    CUDA_FIXED,
    CUDA_LEGACY,
    ENGINES,
    OPENCL_FIXED,
    OPENCL_LEGACY,
    check_engines,
    colour,
    inclusion_model,
    name,
    style,
    survey,
    time_forward_and_gradient,
    where,
)

check_engines()  # the four engines, and the GPU each one runs on

## Choosing the backend

Every propagator of PyFWI (`WavePropagator`, `FWI`, `TimeLapse`) reads its backend from `inpa["backend"]` when
it is created (or, when `inpa` does not say, from the environment variable `PYFWI_BACKEND`). Here is one
forward modelling written out in full, the way the examples of PyFWI write it, run twice: once on OpenCL,
once on CUDA. `cuda:strict` asks for CUDA or an error; plain `cuda` would quietly fall back to OpenCL if CUDA
could not run.

The numeric mode is not set, so each backend runs its default; the last column says which.

In [ ]:
import time

import PyFWI.acquisition as acq
import PyFWI.wave_propagation as wave

nz, nx, dh, ns = 100, 100, 7.0, 4  # a 100 x 100 model with 7 m cells, 4 sources
inpa = {"ns": ns, "sdo": 4, "fdom": 15, "dh": dh, "dt": 0.004, "acq_type": 1, "t": 0.6,
        "npml": 20, "pmlR": 1e-5, "pml_dir": 2, "rec_dis": dh}
src_loc, rec_loc, _, n_well_rec = acq.acq_parameters(ns, dh, nx * dh, nz * dh, dh, inpa["sdo"], acq_type=1)
src_loc[:, 1] -= dh  # the sources one row above the receivers (see comparison.survey)
src = acq.Source(src_loc, dh, inpa["dt"])
src.Ricker(inpa["fdom"])
model = inclusion_model(nz, nx)

for backend in ["opencl", "cuda:strict"]:
    inpa["backend"] = backend  # the only line that changes
    W = wave.WavePropagator(inpa, src, rec_loc, (nz, nx), components=0, n_well_rec=n_well_rec, chpr=0)
    W.forward_modeling(model, show=False)  # the first run compiles the kernels: not timed
    start = time.perf_counter()
    data = W.forward_modeling(model, show=False)
    print(f"{backend:12s} {time.perf_counter() - start:6.3f} s   {where(W)}")

That is already the difference a user sees: the same script, a different backend. But this compares OpenCL in
the `legacy` mode with CUDA in the `fixed` mode, two different calculations. The rest of the notebook times the
four engines one by one, to separate what the backend changes from what the numeric mode changes.

## Settings

`SIZES` are the models to time: the number of grid points (`nz` × `nx`, 7 m apart), of sources (`ns`) and the
length of the recording (`t`, in seconds), long enough for the waves to cross the model. A larger model takes
longer on OpenCL: the large one about a minute and a half. `REPEATS`: each time is the median of this many runs,
after one run that is not timed (it compiles the kernels and warms the GPU up).

In [ ]:
SIZES = {
    "small": {"nz": 100, "nx": 100, "ns": 4, "t": 0.6},  # the example of the PyFWI documentation
    "medium": {"nz": 200, "nx": 200, "ns": 4, "t": 1.0},
    "large": {"nz": 300, "nx": 300, "ns": 2, "t": 1.4},
}
REPEATS = 3

## Timing the four engines

For every size, the observed data come from the true model (`inclusion_model`: a faster disc in a uniform
medium) and the forward modelling and the gradient are those of a smoothed model, as in one iteration of an
inversion. Each engine has its own propagator; nothing else differs.

In [ ]:
times = {}  # times[size][engine] = {"forward": seconds, "gradient": seconds, ...}
for size, p in SIZES.items():
    problem = survey(p["nz"], p["nx"], p["ns"], p["t"])
    true, start = inclusion_model(p["nz"], p["nx"]), inclusion_model(p["nz"], p["nx"], smooth=1)
    times[size] = {}
    for engine in ENGINES:
        t = times[size][engine] = time_forward_and_gradient(problem, engine, true, start, REPEATS)
        print(f"{size:7s} {name(engine):15s} forward {t['forward']:7.3f} s   gradient {t['gradient']:7.3f} s")

## How many times faster

Three comparisons, each the time on OpenCL divided by the time on CUDA (2 means CUDA takes half the time):

* **the same mathematics in the `fixed` mode**: both backends compute the same thing, CUDA with kernels written
  again for the GPU;
* **the same mathematics in the `legacy` mode**: both backends run the very same kernel files, those of PyFWI
  0.1.10, compiled for OpenCL or for CUDA;
* **each backend's default**: what a user gets without setting anything.

In [ ]:
COMPARISONS = {  # title: (OpenCL engine, CUDA engine)
    "Same mathematics, fixed mode": (OPENCL_FIXED, CUDA_FIXED),
    "Same mathematics, legacy mode": (OPENCL_LEGACY, CUDA_LEGACY),
    "Each backend's default": (OPENCL_LEGACY, CUDA_FIXED),
}
CALCULATIONS = {"forward": "forward modelling", "gradient": "gradient"}


def speedup(size, calculation, opencl, cuda):
    return times[size][opencl][calculation] / times[size][cuda][calculation]


print(f"{'CUDA is ... times faster':28s}" + "".join(f"{title:>32s}" for title in COMPARISONS))
for size in times:
    for calculation, text in CALCULATIONS.items():
        print(f"{size + ', ' + text:28s}" + "".join(f"{speedup(size, calculation, *pair):31.1f}x"
                                                    for pair in COMPARISONS.values()))

In [ ]:
labels = [f"{size}, {text}" for size in times for text in CALCULATIONS.values()]
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, (title, (opencl, cuda)) in zip(axes, COMPARISONS.items()):
    values = [speedup(size, c, opencl, cuda) for size in times for c in CALCULATIONS]
    style(ax, grid_axis="x")
    y = np.arange(len(values))
    ax.barh(y, values, color=colour(cuda), height=0.6)
    for i, value in zip(y, values):
        ax.text(value, i, f" {value:.1f}x", va="center", fontsize=9)
    ax.axvline(1, color="black", linewidth=1)  # 1: as fast as OpenCL
    ax.set_xlim(0, max(values) * 1.25)
    ax.set_title(title, loc="left", fontsize=10)
    ax.set_xlabel("times faster on CUDA (1 = as fast as OpenCL)", fontsize=9)
axes[0].set_yticks(y, labels)
axes[0].invert_yaxis()  # the first label on top
fig.suptitle("Same GPU: how many times faster CUDA is than OpenCL", x=0.01, ha="left")
fig.tight_layout()
plt.show()

And the times themselves, on a logarithmic scale (each step of the scale is ten times longer). The colour is the
backend (orange OpenCL, blue CUDA), the hatching the `legacy` mode:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
width = 0.2
for ax, (calculation, text) in zip(axes, CALCULATIONS.items()):
    style(ax)
    for e, engine in enumerate(ENGINES):
        x = np.arange(len(times)) + (e - 1.5) * width
        values = [times[size][engine][calculation] for size in times]
        legacy = engine.numerics == "legacy"
        ax.bar(x, values, width, color=colour(engine), alpha=0.55 if legacy else 1, hatch="///" if legacy else None,
               edgecolor="white", label=name(engine))
    ax.set_yscale("log")
    ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:g} s"))
    ax.set_xticks(np.arange(len(times)), [f"{size} model" for size in times])
    ax.set_ylabel("seconds (log scale; lower is better)", fontsize=9)
    ax.set_title(text.capitalize(), loc="left", fontsize=10)
axes[0].legend(frameon=False, fontsize=9)
fig.tight_layout()
plt.show()

## What these numbers say

* **With the same mathematics in the `fixed` mode, CUDA is about ten times faster or more.** On the RTX 3060 of the
  project the final document measured 8.8× to 22×, the largest gain for the forward modelling of the small model
  ([notebook 4](4_time_per_step.ipynb) explains why).
* **With the same kernel files (the `legacy` mode) it is only a little faster** (1.17× to 1.48× in the final
  document). CUDA is not a faster language: the very same program, compiled for CUDA instead of OpenCL, runs at
  about the same speed. The `legacy` mode keeps the kernels of PyFWI 0.1.10 on purpose, so that its results stay
  those of the original.
* **The gain comes from the kernels written again for the GPU** in the `fixed` mode: fewer orders sent from the
  computer to the GPU at every time step, the steps replayed by the GPU from a recording (*CUDA graphs*), the
  snapshots of the forward simulation kept in the memory of the GPU instead of copied to the computer, and
  kernels that read less memory. [Notebook 4](4_time_per_step.ipynb) shows where the time goes, and
  [notebook 5](5_memory.ipynb) the memory.
* **By default** (OpenCL `legacy`, CUDA `fixed`) the gain is about the same as in the `fixed` mode, but the two
  backends then compute slightly different things: [notebook 2](2_same_results.ipynb) shows by how much.